In [ ]:
# Copyright 2025 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Model Garden을 이용한 빠른 시작 - MedGemma

<table><tbody><tr>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2FGoogle-Health%2Fmedgemma%2Fmain%2Fnotebooks%2Fquick_start_with_model_garden.ipynb">
      <img alt="Google Cloud Colab Enterprise logo" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" width="32px"><br> Colab Enterprise에서 실행
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/google-health/medgemma/blob/main/notebooks/quick_start_with_model_garden.ipynb">
      <img alt="Google Colab logo" src="https://www.tensorflow.org/images/colab_logo_32px.png" width="32px"><br> Google Colab에서 실행
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/google-health/medgemma/blob/main/notebooks/quick_start_with_model_garden.ipynb">
      <img alt="GitHub logo" src="https://github.githubassets.com/assets/GitHub-Mark-ea2971cee799.png" width="32px"><br> GitHub에서 보기
    </a>
  </td>
</tr></tbody></table>

## 개요

이 노트북은 Vertex AI에서 모델 추론을 가져오는 두 가지 방법을 사용하여 의학 텍스트 및 이미지에서 응답을 생성하기 위해 MedGemma를 사용하는 방법을 보여줍니다:

* **Online inferences (온라인 추론)**은 Model Garden에서 배포된 엔드포인트(endpoint) 로 이루어지는 동기식 요청이며 낮은 대기 시간(low latency)으로 제공됩니다. 모델 출력이 프로덕션에서 사용되는 경우 온라인 추론이 유용합니다. 온라인 추론 비용은 가상 머신이 추론 요청을 처리하기 위해 활성 상태(배포된 모델이 있는 엔드포인트)에서 대기하는 시간을 기준으로 합니다.

* **Batch inferences (배치 추론)**은 단일 작업에 지정된 정해진 수의 모델 입력에서 실행되는 비동기식 요청입니다. Model Garden에서 배포된 엔드포인트를 사용하지 않고 업로드된 모델에 직접 요청이 이루어집니다. 학습에 사용하기 위해 대량의 입력에 대한 누적 추론 결과를 얻고자 하고 짧은 대기 시간이 필요하지 않은 경우 배치 추론이 유용합니다. 배치 추론 비용은 가상 머신이 추론 작업을 실행하는 데 소요된 시간을 기준으로 합니다.

Vertex AI를 사용하면 모델을 쉽게 제공하고 배포할 수 있습니다. [Vertex AI](https://cloud.google.com/vertex-ai/docs/start/introduction-unified-platform)에 대해 자세히 알아보세요.

### 목표

- Vertex AI 엔드포인트에 MedGemma를 배포하고 온라인 추론 얻기.
- MedGemma를 Vertex AI 모델 레지스트리(Model Registry)에 업로드하고 배치 추론 얻기.

### 비용

이 튜토리얼은 청구 대상인 다음과 같은 Google Cloud 구성 요소를 사용합니다:

* Vertex AI
* Cloud Storage

[Vertex AI 가격 책정(pricing)](https://cloud.google.com/vertex-ai/pricing), [Cloud Storage 가격 책정](https://cloud.google.com/storage/pricing)에 대해 알아보고, [가격 계산기(Pricing Calculator)](https://cloud.google.com/products/calculator/)를 사용하여 예상 사용량을 기준으로 비용 예상치를 생성해 보세요.

## 시작하기 전에

In [ ]:
# @title 패키지 가져오기 및 공통 함수 정의

import datetime
import importlib
import json
import os
import requests
import sys
import uuid

from google.cloud import aiplatform, storage
from IPython.display import Image as IPImage, display, Markdown
import google.auth
import openai

if not os.path.isdir("vertex-ai-samples"):
    ! git clone https://github.com/GoogleCloudPlatform/vertex-ai-samples.git

common_util = importlib.import_module(
    "vertex-ai-samples.community-content.vertex_model_garden.model_oss.notebook_util.common_util"
)

models, endpoints = {}, {}

In [ ]:
# @title Google Cloud 인증 (Google Colab에서만 필요)

google_colab = "google.colab" in sys.modules and not os.environ.get("VERTEX_PRODUCT") == "COLAB_ENTERPRISE"

if google_colab:
    from google.colab import auth
    # 사용자 계정으로 로그인하고 액세스를 승인하라는
    # 팝업이 나타납니다.
    auth.authenticate_user()

In [ ]:
# @title Google Cloud 환경 설정

# @markdown #### 전제 조건
# @markdown 1. Vertex AI를 시작하려면 Vertex AI API가 활성화된 기존 Google Cloud 프로젝트가 필요합니다. [프로젝트 및 개발 환경 설정](https://docs.cloud.google.com/vertex-ai/docs/start/cloud-environment) 가이드를 참조하세요.
# @markdown 2. 프로젝트에 대해 [결제가 활성화되어 있는지](https://cloud.google.com/billing/docs/how-to/modify-project) 확인하세요.
# @markdown 3. 다음과 같은 필수 역할(roles)이 있는지 확인하세요:
# @markdown    - [Vertex AI 사용자](https://cloud.google.com/vertex-ai/docs/general/access-control?_gl=1*j68dlr*_ga*MTExMjk3OTcwMy4xNzY4MzcyMzc2*_ga_WH2QY8WWF5*czE3Njg0MzAxNTYkbzUkZzEkdDE3Njg0MzU4NjUkajYwJGwwJGgw#aiplatform.user) (`roles/aiplatform.user`)
# @markdown    - [Colab Enterprise 사용자](https://cloud.google.com/vertex-ai/docs/general/access-control?_gl=1*j68dlr*_ga*MTExMjk3OTcwMy4xNzY4MzcyMzc2*_ga_WH2QY8WWF5*czE3Njg0MzAxNTYkbzUkZzEkdDE3Njg0MzU4NjUkajYwJGwwJGgw#aiplatform.colabEnterpriseUser) (`roles/aiplatform.colabEnterpriseUser`), Colab Enterprise에서 이 노트북을 실행할 의향이 있는 경우 (Google Colab에는 필요하지 않음)
# @markdown 4. Compute Engine API가 활성화되어 있는지 아니면 API를 활성화할 [Service Usage 관리자](https://cloud.google.com/iam/docs/understanding-roles#serviceusage.serviceUsageAdmin) (`roles/serviceusage.serviceUsageAdmin`) 역할이 있는지 확인하세요.

# @markdown 이 섹션은 Google Cloud 프로젝트 및 리전(region)을 설정하고, Vertex AI 및 Compute Engine API를 활성화(아직 활성화되지 않은 경우)하고, Vertex AI API를 초기화합니다.
# @markdown
# @markdown 아래에 Google Cloud 프로젝트 ID 및 리전을 설정하세요. Colab Enterprise에서 이 노트북을 실행하는 경우 필드를 비워두어 노트북 환경의 기본 프로젝트 ID 및 리전을 사용할 수 있습니다.
PROJECT_ID = ""  # @param {type: "string", placeholder: "e.g. your-project"}

# 환경에서 기본 프로젝트 ID 가져오기
if not PROJECT_ID:
    PROJECT_ID = os.environ["GOOGLE_CLOUD_PROJECT"]
os.environ["CLOUDSDK_CORE_PROJECT"] = PROJECT_ID

REGION = ""  # @param {type: "string", placeholder: "e.g. us-central1"}

# 환경에서 작업 실행을 위한 기본 리전 가져오기
if not REGION:
    REGION = os.environ["GOOGLE_CLOUD_REGION"]

# 아직 활성화되지 않은 경우 Vertex AI 및 Compute Engine API 활성화
print("Enabling Vertex AI API.")
! gcloud services enable aiplatform.googleapis.com
print("Enabling Compute Engine API.")
! gcloud services enable compute.googleapis.com

# Vertex AI API 초기화
print("Initializing Vertex AI API.")
aiplatform.init(project=PROJECT_ID, location=REGION, api_transport="rest")

## 온라인 추론 얻기

In [ ]:
# @title Vertex AI 엔드포인트 설정

# @markdown [온라인 추론(online inferences)](https://cloud.google.com/vertex-ai/docs/predictions/get-online-predictions)을 얻으려면 Model Garden에서 배포된 MedGemma [Vertex AI 엔드포인트](https://cloud.google.com/vertex-ai/docs/general/deployment)가 필요합니다. 아직 배포하지 않은 경우 [MedGemma 모델 카드](https://console.cloud.google.com/vertex-ai/publishers/google/model-garden/medgemma)로 이동하여 **"Deploy model"**을 클릭하여 모델을 배포하세요.
# @markdown
# @markdown **참고:** 이 노트북의 예제들은 인스트럭션 튜닝(instruction-tuned) 변형 모델들과 함께 사용하기 위한 목적입니다. 이 노트북을 실행하려면 반드시 인스트럭션 튜닝된 모델 변형을 사용하세요.
# @markdown
# @markdown 이 섹션에서는 Model Garden에서 배포한 Vertex AI 엔드포인트 리소스를 가져와 온라인 추론에 사용합니다.
# @markdown
# @markdown 아래에 엔드포인트 ID와 리전을 채우세요. 배포된 엔드포인트는 [Vertex AI endpoints 페이지](https://console.cloud.google.com/vertex-ai/online-prediction/endpoints)에서 찾을 수 있습니다.

ENDPOINT_ID = ""  # @param {type: "string", placeholder: "e.g. 123456789"}
ENDPOINT_REGION = ""  # @param {type: "string", placeholder: "e.g. us-central1"}

endpoints["endpoint"] = aiplatform.Endpoint(
    endpoint_name=ENDPOINT_ID,
    location=ENDPOINT_REGION,
)

# 엔드포인트 이름을 사용하여 적절한 모델 변형 및 설정을 사용하고 있는지
# 확인합니다. 이러한 확인은 Model Garden 배포 설정의 기본
# 엔드포인트 이름을 기반으로 합니다.
ENDPOINT_NAME = endpoints["endpoint"].display_name
if "pt" in ENDPOINT_NAME:
    raise ValueError(
        "The examples in this notebook are intended to be used with "
        "instruction-tuned variants. Please use an instruction-tuned model."
    )

# @markdown ---
# @markdown [전용(dedicated) 엔드포인트](https://cloud.google.com/vertex-ai/docs/predictions/choose-endpoint-type)를 사용하는 경우 `use_dedicated_endpoint`를 설정하세요(Model Garden 배포의 경우 기본적으로 `True`). 다른 모든 엔드포인트 유형에 대해서는 이 옵션을 선택 취소합니다.

use_dedicated_endpoint = True  # @param {type: "boolean"}

# @markdown ---
# @markdown MedGemma 1.5 4B 및 MedGemma 27B 변형은 thinking 기능을 지원합니다. 활성화된 경우 모델이 최종 응답 전에 `<unused94>` 및 `<unused95>`라는 두 가지 특수 토큰 사이에 추론 과정을 출력하도록 프롬프트에 system instruction이 추가됩니다. **참고:** 이 노트북은 다중 모드(multimodal) 또는 텍스트 전용(text-only) 작업에 대해 thinking 기능을 활성화하는 방법을 보여 주지만, 일부 활용 사례에는 도움이 되지 않을 수 있습니다. Thinking 기능은 어려운 텍스트 전용 작업에서는 결과를 향상시킬 수 있지만 더 많은 토큰을 생성하게 됩니다. 모델의 성능을 철저히 검증하고 thinking 기능을 활성화할 때 추론 시간과 비용을 평가하세요.
is_thinking = False  # @param {type: "boolean"}
THINKING_VARIANTS = [
    "medgemma-1.5-4b-it",
    "medgemma-27b-it",
    "medgemma-27b-text-it",
]
if is_thinking and not any(variant in ENDPOINT_NAME for variant in THINKING_VARIANTS):
    is_thinking = False
    print(
        "Note: Thinking is enabled for a non-thinking variant. Setting "
        "`is_thinking` to `False`."
    )

### 이미지 및 텍스트 추론 실행

이 섹션은 다중 모드(multimodal) 변형들을 사용하여 이미지 기반 작업에서 추론을 실행하는 것을 보여줍니다.

**참고:** 27B 텍스트 전용 변형을 선택한 경우 [텍스트 전용 추론 실행](#scrollTo=lRQDWe2znWn7)으로 진행하세요.

In [ ]:
# @title #### 이미지 및 텍스트 입력 지정

# 다중 모드 변형을 사용하고 있는지 확인하세요
if "text" in ENDPOINT_NAME:
    raise ValueError(
        "You are using a text-only variant which does not support multimodal"
        " inputs. Proceed to the 'Run inference on text only' section."
    )

system_instruction = "You are an expert radiologist."
prompt = "Describe this X-ray" # @param {type: "string"}
image_url = "https://upload.wikimedia.org/wikipedia/commons/c/c8/Chest_Xray_PA_3-8-2010.png" # @param {type: "string"}
! wget -nc -q {image_url}
image_filename = os.path.basename(image_url)

In [ ]:
# @title #### 대화 형식 (Format conversation)

if is_thinking:
    system_instruction = f"SYSTEM INSTRUCTION: think silently if needed. {system_instruction}"
    max_tokens = 1500
else:
    max_tokens = 500

messages = [
    {
        "role": "system",
        "content": [{"type": "text", "text": system_instruction}]
    },
    {
        "role": "user",
        "content": [
            {"type": "text", "text": prompt},
            {"type": "image_url", "image_url": {"url": image_url}}
        ]
    }
]

In [ ]:
# @title #### Vertex AI SDK를 사용하여 응답 생성

# @markdown 이 섹션은 Vertex AI 엔드포인트에 [온라인 추론](https://cloud.google.com/vertex-ai/docs/predictions/get-online-predictions) 요청을 보내는 방법을 보여줍니다.

# @markdown "Show Code"를 클릭하여 자세한 내용을 확인하세요.

display(Markdown(f"---\n\n**[ User ]**\n\n{prompt}"))
display(IPImage(filename=image_filename, height=300))

instances = [
    {
        "@requestFormat": "chatCompletions",
        "messages": messages,
        "max_tokens": max_tokens,
        "temperature": 0
    },
]

response = endpoints["endpoint"].predict(
    instances=instances, use_dedicated_endpoint=use_dedicated_endpoint
).predictions["choices"][0]["message"]["content"]

if is_thinking:
    thought, response = response.split("<unused95>")
    thought = thought.replace("<unused94>thought\n", "")
    display(Markdown(f"---\n\n**[ MedGemma thinking trace ]**\n\n{thought}"))
display(Markdown(f"---\n\n**[ MedGemma ]**\n\n{response}\n\n---"))

In [ ]:
# @title #### OpenAI SDK를 사용하여 응답 생성

# @markdown 이 섹션은 OpenAI SDK를 사용하여 엔드포인트에 [채팅 완료(chat completions)](https://platform.openai.com/docs/api-reference/chat) 요청을 보내는 방법을 보여줍니다.

# @markdown "Show Code"를 클릭하여 자세한 내용을 확인하세요.

if "dicom" in ENDPOINT_NAME:
    raise NotImplementedError(
        "MedGemma DICOM endpoints are currently not OpenAI-compatible. Use the "
        "Vertex AI SDK to run inference."
    )

display(Markdown(f"---\n\n**[ User ]**\n\n{prompt}"))
display(IPImage(filename=image_filename, height=300))

creds, project = google.auth.default()
auth_req = google.auth.transport.requests.Request()
creds.refresh(auth_req)

ENDPOINT_RESOURCE_NAME = endpoints["endpoint"].resource_name

if use_dedicated_endpoint:
    DEDICATED_ENDPOINT_DNS = endpoints["endpoint"].gca_resource.dedicated_endpoint_dns
    BASE_URL = f"https://{DEDICATED_ENDPOINT_DNS}/v1beta1/{ENDPOINT_RESOURCE_NAME}"
else:
    BASE_URL = f"https://{ENDPOINT_REGION}-aiplatform.googleapis.com/v1beta1/{ENDPOINT_RESOURCE_NAME}"

client = openai.OpenAI(base_url=BASE_URL, api_key=creds.token)

model_response = client.chat.completions.create(
    model="",
    messages=messages,
    max_completion_tokens=max_tokens,
    temperature=0,
)
response = model_response.choices[0].message.content

if is_thinking:
    thought, response = response.split("<unused95>")
    thought = thought.replace("<unused94>thought\n", "")
    display(Markdown(f"---\n\n**[ MedGemma thinking trace ]**\n\n{thought}"))
display(Markdown(f"---\n\n**[ MedGemma ]**\n\n{response}\n\n---"))

### 텍스트 전용 추론 실행

이 섹션에서는 텍스트 기반 작업에 대한 추론 실행을 보여줍니다.

In [ ]:
# @title #### 텍스트 프롬프트 지정

system_instruction = "You are a helpful medical assistant."
prompt = "How do you differentiate bacterial from viral pneumonia?"  # @param {type:"string"}

if is_thinking:
    system_instruction = f"SYSTEM INSTRUCTION: think silently if needed. {system_instruction}"
    max_tokens = 2000
else:
    max_tokens = 1000

messages = [
    {
        "role": "system",
        "content": system_instruction
    },
    {
        "role": "user",
        "content": prompt
    }
]

In [ ]:
# @title #### Vertex AI SDK를 사용하여 응답 생성

# @markdown 이 섹션은 Vertex AI 엔드포인트에 [온라인 추론](https://cloud.google.com/vertex-ai/docs/predictions/get-online-predictions) 요청을 보내는 방법을 보여줍니다.

# @markdown "Show Code"를 클릭하여 자세한 내용을 확인하세요.

display(Markdown(f"---\n\n**[ User ]**\n\n{prompt}\n\n---"))

instances = [
    {
        "@requestFormat": "chatCompletions",
        "messages": messages,
        "max_tokens": max_tokens,
        "temperature": 0
    },
]

response = endpoints["endpoint"].predict(
    instances=instances, use_dedicated_endpoint=use_dedicated_endpoint
).predictions["choices"][0]["message"]["content"]

if is_thinking:
    thought, response = response.split("<unused95>")
    thought = thought.replace("<unused94>thought\n", "")
    display(Markdown(f"**[ MedGemma thinking trace ]**\n\n{thought}\n\n---"))
display(Markdown(f"**[ MedGemma ]**\n\n{response}\n\n---"))

In [ ]:
# @title #### OpenAI SDK를 사용하여 응답 생성

# @markdown 이 섹션은 OpenAI SDK를 사용하여 엔드포인트에 [채팅 완료(chat completions)](https://platform.openai.com/docs/api-reference/chat) 요청을 보내는 방법을 보여줍니다.

# @markdown "Show Code"를 클릭하여 자세한 내용을 확인하세요.

if "dicom" in ENDPOINT_NAME:
    raise NotImplementedError(
        "MedGemma DICOM endpoints are currently not OpenAI-compatible. Use the "
        "Vertex AI SDK to run inference."
    )

display(Markdown(f"---\n\n**[ User ]**\n\n{prompt}\n\n---"))

creds, project = google.auth.default()
auth_req = google.auth.transport.requests.Request()
creds.refresh(auth_req)

ENDPOINT_RESOURCE_NAME = endpoints["endpoint"].resource_name

if use_dedicated_endpoint:
    DEDICATED_ENDPOINT_DNS = endpoints["endpoint"].gca_resource.dedicated_endpoint_dns
    BASE_URL = f"https://{DEDICATED_ENDPOINT_DNS}/v1beta1/{ENDPOINT_RESOURCE_NAME}"
else:
    BASE_URL = f"https://{ENDPOINT_REGION}-aiplatform.googleapis.com/v1beta1/{ENDPOINT_RESOURCE_NAME}"

client = openai.OpenAI(base_url=BASE_URL, api_key=creds.token)

model_response = client.chat.completions.create(
    model="",
    messages=messages,
    max_completion_tokens=max_tokens,
    temperature=0,
)
response = model_response.choices[0].message.content

if is_thinking:
    thought, response = response.split("<unused95>")
    thought = thought.replace("<unused94>thought\n", "")
    display(Markdown(f"**[ MedGemma thinking trace ]**\n\n{thought}\n\n---"))
display(Markdown(f"**[ MedGemma ]**\n\n{response}\n\n---"))

### 스트림(Stream) 응답

In [ ]:
# @markdown 이 섹션은 스트리밍 응답을 보여주며, 이를 통해 전체 응답이 생성되는 동안 모델 출력을 청크(chunk) 단위로 표시하거나 처리할 수 있습니다.
# @markdown
# @markdown Vertex AI 엔드포인트(`use_vertex`를 `True`로 설정)를 직접 사용하거나 (대안적으로) OpenAI SDK(`False`로 설정)를 사용하여 응답을 직접 스트리밍할 수 있습니다.

# @markdown "Show Code"를 클릭하여 자세한 내용을 확인하세요.

if "dicom" in ENDPOINT_NAME:
    raise NotImplementedError(
        "MedGemma DICOM endpoints currently do not support streaming."
    )

use_vertex = True  # @param {type: "boolean"}

prompt = "How do you differentiate bacterial from viral pneumonia?"
display(Markdown(f"---\n\n**[ User ]**\n\n{prompt}\n\n---\n\n**[MedGemma]**"))

messages = [
    {
        "role": "system",
        "content": "You are a helpful medical assistant."
    },
    {
        "role": "user",
        "content": prompt
    }
]
max_tokens = 1500

creds, project = google.auth.default()
auth_req = google.auth.transport.requests.Request()
creds.refresh(auth_req)

ENDPOINT_RESOURCE_NAME = endpoints["endpoint"].resource_name

if use_dedicated_endpoint:
    DEDICATED_ENDPOINT_DNS = endpoints["endpoint"].gca_resource.dedicated_endpoint_dns
    BASE_URL = f"https://{DEDICATED_ENDPOINT_DNS}/v1beta1/{ENDPOINT_RESOURCE_NAME}"
else:
    BASE_URL = f"https://{ENDPOINT_REGION}-aiplatform.googleapis.com/v1beta1/{ENDPOINT_RESOURCE_NAME}"

if use_vertex:
    endpoint_url = f"{BASE_URL}/chat/completions"
    request = {
        "messages": messages,
        "max_tokens": max_tokens,
        "temperature": 0,
        "stream": True
    }
    with requests.post(
        endpoint_url,
        headers={"Authorization": f"Bearer {creds.token}"},
        json=request,
        stream=True,
    ) as r:
        for chunk in r.iter_lines(chunk_size=8192):
            if chunk:
                chunk = chunk.decode("utf-8").removeprefix("data:").strip()
                if chunk == "[DONE]":
                    break
                chunk = json.loads(chunk)
                print(chunk["choices"][0]["delta"]["content"], sep=" ", end="")
else:
    client = openai.OpenAI(base_url=BASE_URL, api_key=creds.token)
    response = client.chat.completions.create(
        model="",
        messages=messages,
        max_completion_tokens=max_tokens,
        temperature=0,
        stream=True,
    )
    for chunk in response:
        print(chunk.choices[0].delta.content, sep=" ", end="")

## 배치 추론 얻기

In [ ]:
# @title MedGemma 접근 권한 얻기

# @markdown 추론 컨테이너는 Hugging Face Hub에서 직접 모델을 로드합니다.
# @markdown
# @markdown MedGemma 모델에 대한 접근을 활성화하려면 Hugging Face 사용자 액세스 토큰(User Access Token)을 제공해야 합니다. [Hugging Face 문서](https://huggingface.co/docs/hub/en/security-tokens)를 따라 **읽기(read)** 액세스 토큰을 생성하고 아래 `HF_TOKEN` 필드에 지정할 수 있습니다.

HF_TOKEN = ""  # @param {type: "string", placeholder: "Hugging Face User Access Token"}


In [ ]:
# @title Vertex AI 모델 레지스트리(Model Registry)에 모델 업로드

# @markdown [배치 추론](https://cloud.google.com/vertex-ai/docs/predictions/get-batch-predictions)을 얻으려면 사전 빌드된 MedGemma 모델을 먼저 [Vertex AI Model Registry](https://cloud.google.com/vertex-ai/docs/model-registry/introduction)에 업로드해야 합니다. 배치 추론 요청은 엔드포인트에 배포하지 않고 Model Registry의 모델에 직접 이루어집니다.

# 참고: 27B 변형에 대해서는 배치 추론이 작동하지 않을 수 있으므로 드롭다운에 포함되지 않았습니다.
MODEL_VARIANT = "4b-it"  # @param ["4b-it"]

MODEL_ID = f"medgemma-{MODEL_VARIANT}"

# 사전 빌드된 서빙 도커(docker) 이미지.
SERVE_DOCKER_URI = "us-docker.pkg.dev/vertex-ai/vertex-vision-model-garden-dockers/pytorch-vllm-serve:20250430_0916_RC00_maas"

# 이 노트북은 데모용으로 Nvidia L4 GPU를 사용합니다.
# Vertex AI 배치 추론용 컴퓨팅 구성에 대한 자세한 내용은
# https://cloud.google.com/vertex-ai/docs/predictions/configure-compute#batch_prediction 를 참조하세요.
if "4b" in MODEL_ID:
    accelerator_type = "NVIDIA_L4"
    machine_type = "g2-standard-24"
    accelerator_count = 2
elif "27b" in MODEL_ID:
    accelerator_type = "NVIDIA_L4"
    machine_type = "g2-standard-48"
    accelerator_count = 4
else:
    raise ValueError(
        f"Recommended machine settings not found for model: {MODEL_ID}."
    )


def upload_model(
    model_name: str,
    model_id: str,
    accelerator_count: int = 1,
    gpu_memory_utilization: float = 0.95,
    max_model_len: int = 32768,
    max_num_seqs: int = 16,
    max_images: int = 16,
) -> aiplatform.Model:

    vllm_args = [
        "python",
        "-m",
        "vllm.entrypoints.api_server",
        "--host=0.0.0.0",
        "--port=8080",
        f"--model={model_id}",
        f"--tensor-parallel-size={accelerator_count}",
        "--swap-space=16",
        f"--gpu-memory-utilization={gpu_memory_utilization}",
        f"--max-model-len={max_model_len}",
        f"--max-num-seqs={max_num_seqs}",
        "--enable-chunked-prefill",
        "--disable-log-stats",
    ]

    if "text" not in model_id:
        vllm_args.extend([
            f"--limit_mm_per_prompt='image={max_images}'",
            "--mm-processor-kwargs='{\"do_pan_and_scan\": true}'"
        ])


    env_vars = {
        "MODEL_ID": model_id,
        "DEPLOY_SOURCE": "notebook",
        "VLLM_USE_V1": "0",
        "HF_TOKEN": HF_TOKEN,
    }

    model = aiplatform.Model.upload(
        display_name=model_name,
        serving_container_image_uri=SERVE_DOCKER_URI,
        serving_container_args=vllm_args,
        serving_container_ports=[8080],
        serving_container_predict_route="/generate",
        serving_container_health_route="/ping",
        serving_container_environment_variables=env_vars,
    )
    return model


models["model"] = upload_model(
    model_name=common_util.get_job_name_with_datetime(prefix=MODEL_ID),
    model_id=f"google/{MODEL_ID}",
    accelerator_count=accelerator_count,
)

In [ ]:
# @title Google Cloud 리소스 설정

# @markdown 이 섹션에서는 배치 추론 입력 및 출력을 저장하기 위한 [Cloud Storage 버킷(bucket)](https://cloud.google.com/storage/docs/creating-buckets)을 설정하고 배치 추론 작업을 실행하는 데 사용될 [Compute Engine 기본 서비스 계정(default service account)](https://cloud.google.com/compute/docs/access/service-accounts#default_service_account)을 가져옵니다.

# @markdown 1. 다음과 같은 필수 역할(roles)이 있는지 확인하세요:
# @markdown - Cloud Storage 버킷을 생성하고 사용하기 위한 [Storage 관리자](https://cloud.google.com/iam/docs/understanding-roles#storage.admin) (`roles/storage.admin`)
# @markdown - 프로젝트 또는 Compute Engine 기본 서비스 계정의 [서비스 계정 사용자](https://cloud.google.com/iam/docs/understanding-roles#iam.serviceAccountUser) (`roles/iam.serviceAccountUser`)

# @markdown 2. Cloud Storage 버킷을 설정합니다.
# @markdown - 새 버킷이 자동으로 생성됩니다.
# @markdown - [선택] 기존 버킷을 사용하려면 `gs://` 버킷 URI를 지정하세요. 지정된 Cloud Storage 버킷은 노트북이 시작된 지역과 동일한 리전에 있어야 합니다. 멀티 리전 버킷(예: "us")은 멀티 리전 범위가 포함하는 단일 리전(예: "us-central1")과 일치하는 것으로 간주되지 않습니다.

BUCKET_URI = ""  # @param {type: "string", placeholder: "[Optional] Cloud Storage bucket URI"}

# 배치 추론 아티팩트 저장을 위한 Cloud Storage 버킷.
# 이 노트북의 목적에 맞게 고유한 버킷이 생성됩니다.
# 고유한 GCS 버킷을 사용하는 것을 선호한다면 위의 BUCKET_URI 값을 직접 변경하세요.
if BUCKET_URI is None or BUCKET_URI.strip() == "":
    now = datetime.datetime.now().strftime("%Y%m%d%H%M%S")
    BUCKET_URI = f"gs://{PROJECT_ID}-tmp-{now}-{str(uuid.uuid4())[:4]}"
    BUCKET_NAME = "/".join(BUCKET_URI.split("/")[:3])
    ! gcloud storage buckets create --location {REGION} {BUCKET_URI}
else:
    assert BUCKET_URI.startswith("gs://"), "BUCKET_URI must start with `gs://`."
    BUCKET_NAME = "/".join(BUCKET_URI.split("/")[:3])
    shell_output = ! gcloud storage buckets describe {BUCKET_NAME} | grep "location:" | sed "s/location://"
    bucket_region = shell_output[0].strip().lower()
    if bucket_region != REGION:
        raise ValueError(
            f"Bucket region {bucket_region} is different from notebook region {REGION}"
        )
print(f"Using this Cloud Storage Bucket: {BUCKET_URI}")

# 추론 컨테이너 실행에 사용되는 서비스 계정.
# Compute Engine 기본 서비스 계정을 가져옵니다. 자신만의 사용자 지정(custom)
# 서비스 계정을 사용하는 것을 선호한다면 아래의 SERVICE_ACCOUNT 값을 변경하세요.
shell_output = ! gcloud projects describe $PROJECT_ID
project_number = shell_output[-1].split(":")[1].strip().replace("'", "")
SERVICE_ACCOUNT = f"{project_number}-compute@developer.gserviceaccount.com"
print("Using this service account:", SERVICE_ACCOUNT)

### 예측 (Predict)

출력을 생성하기 위해 텍스트 프롬프트와 이미지가 있는 입력 인스턴스 목록을 지정하는 [JSON Lines](https://jsonlines.org/) 파일을 사용하여 모델에 [배치 추론 요청](https://cloud.google.com/vertex-ai/docs/predictions/get-batch-predictions#request_a_batch_prediction)을 보낼 수 있습니다. 배치 추론 작업 구성에 대한 자세한 내용은 [입력 데이터의 형식 지정](https://cloud.google.com/vertex-ai/docs/predictions/get-batch-predictions#input_data_requirements) 방법과 [컴퓨팅 설정 선택 (choose compute settings)](https://cloud.google.com/vertex-ai/docs/predictions/get-batch-predictions#choose_machine_type_and_replica_count) 방법을 참조하세요.

In [ ]:
# @title 이미지 및 텍스트에서 일괄(batch) 응답 생성

# @markdown 이 섹션은 다중 모드(multimodal) 변형들을 사용하여 이미지 기반 작업에서 일괄 추론을 실행하는 방법을 보여줍니다.
# @markdown
# @markdown **참고:** 27B 텍스트 전용 변형을 선택한 경우 [텍스트 전용 일괄(batch) 응답 생성](#scrollTo=sbMkoiJ161hO)으로 진행하세요.

# @markdown "Show Code"를 클릭하여 자세한 내용을 확인하세요.

# 다중 모드 변형을 사용하고 있는지 확인하세요
if "text" in MODEL_VARIANT:
    raise ValueError(
        "You are using a text-only variant which does not support multimodal "
        "inputs. Please proceed to the 'Generate responses in batch from text "
        "only' section."
    )

batch_predict_instances = [
    {
        "@requestFormat": "chatCompletions",
        "messages": [
            {
                "role": "system",
                "content": [{"type": "text", "text": "You are an expert radiologist."}]
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": "Describe this X-ray"
                    },
                    {
                        "type": "image_url",
                        "image_url": {"url": "https://upload.wikimedia.org/wikipedia/commons/c/c8/Chest_Xray_PA_3-8-2010.png"}
                    }
                ]
            }
        ],
        "max_tokens": 200
    }
]

# JSON Lines 파일에 인스턴스 쓰기
os.makedirs("batch_predict_input", exist_ok=True)
instances_filename = "multimodal_instances.jsonl"
with open(f"batch_predict_input/{instances_filename}", "w") as f:
  for line in batch_predict_instances:
    json_str = json.dumps(line)
    f.write(json_str)
    f.write("\n")

# Cloud Storage로 파일 복사
batch_predict_prefix = f"batch-predict-{MODEL_ID}"
! gcloud storage cp ./batch_predict_input/{instances_filename} {BUCKET_URI}/{batch_predict_prefix}/input/{instances_filename}

batch_predict_job_name = common_util.get_job_name_with_datetime(
    prefix=f"batch-predict-{MODEL_ID}"
)

multimodal_batch_predict_job = models["model"].batch_predict(
    job_display_name=batch_predict_job_name,
    gcs_source=os.path.join(
        BUCKET_URI, batch_predict_prefix, f"input/{instances_filename}"
    ),
    gcs_destination_prefix=os.path.join(
        BUCKET_URI, batch_predict_prefix, "output"
    ),
    machine_type=machine_type,
    accelerator_type=accelerator_type,
    accelerator_count=accelerator_count,
    service_account=SERVICE_ACCOUNT,
)

multimodal_batch_predict_job.wait()

print(multimodal_batch_predict_job.display_name)
print(multimodal_batch_predict_job.resource_name)
print(multimodal_batch_predict_job.state)

In [ ]:
# @title 텍스트 전용(text only)에서 일괄 응답 생성

# @markdown 이 섹션은 텍스트 기반 작업에 대한 일괄 추론 실행을 보여줍니다.

# @markdown "Show Code"를 클릭하여 자세한 내용을 확인하세요.

batch_predict_instances = [
    {
        "@requestFormat": "chatCompletions",
        "messages": [
            {
                "role": "system",
                "content": "You are a helpful medical assistant."
            },
            {
                "role": "user",
                "content": "How do you differentiate bacterial from viral pneumonia?"
            }
        ],
        "max_tokens": 200
    }
]

# JSON Lines 파일에 인스턴스 쓰기
os.makedirs("batch_predict_input", exist_ok=True)
instances_filename = "text_instances.jsonl"
with open(f"batch_predict_input/{instances_filename}", "w") as f:
  for line in batch_predict_instances:
    json_str = json.dumps(line)
    f.write(json_str)
    f.write("\n")

# Cloud Storage로 파일 복사
batch_predict_prefix = f"batch-predict-{MODEL_ID}"
! gcloud storage cp ./batch_predict_input/{instances_filename} {BUCKET_URI}/{batch_predict_prefix}/input/{instances_filename}

batch_predict_job_name = common_util.get_job_name_with_datetime(
    prefix=f"batch-predict-{MODEL_ID}"
)

text_batch_predict_job = models["model"].batch_predict(
    job_display_name=batch_predict_job_name,
    gcs_source=os.path.join(
        BUCKET_URI, batch_predict_prefix, f"input/{instances_filename}"
    ),
    gcs_destination_prefix=os.path.join(
        BUCKET_URI, batch_predict_prefix, "output"
    ),
    machine_type=machine_type,
    accelerator_type=accelerator_type,
    accelerator_count=accelerator_count,
    service_account=SERVICE_ACCOUNT,
)

text_batch_predict_job.wait()

print(text_batch_predict_job.display_name)
print(text_batch_predict_job.resource_name)
print(text_batch_predict_job.state)

In [ ]:
# @title #### 추론 결과 얻기

# @markdown 이 섹션은 출력 Cloud Storage 위치에 있는 JSON Lines 파일(들)에서 [배치 추론 결과를 검색](https://cloud.google.com/vertex-ai/docs/predictions/get-batch-predictions#retrieve_batch_prediction_results)하는 예시를 보여줍니다.

# @markdown "Show Code"를 클릭하여 자세한 내용을 확인하세요.

def download_gcs_files_as_json(gcs_files_prefix):
    """Download specified files from Cloud Storage and convert content to JSON."""
    lines = []
    client = storage.Client()
    bucket = storage.bucket.Bucket.from_string(BUCKET_NAME, client)
    blobs = bucket.list_blobs(prefix=gcs_files_prefix)
    for blob in blobs:
        with blob.open("r") as f:
            for line in f:
                lines.append(json.loads(line))
    return lines


# 첫 번째 배치 추론 작업(다중 모드 입력 보유)에서 결과 얻기
# 다른 배치 추론 작업에서 결과를 얻으려면 이 변수를 교체할 수 있습니다.
batch_predict_job = multimodal_batch_predict_job
batch_predict_output_dir = batch_predict_job.output_info.gcs_output_directory
batch_predict_output_files_prefix = os.path.join(
    batch_predict_output_dir.replace(f"{BUCKET_NAME}/", ""),
    "prediction.results"
)
batch_predict_results = download_gcs_files_as_json(
    gcs_files_prefix=batch_predict_output_files_prefix
)

# 첫 번째 배치 추론 결과 표시
line = batch_predict_results[0]
prediction = line["prediction"]["predictions"]["choices"][0]["message"]["content"]
display(Markdown(prediction))


## 다음 단계

다른 [노트북](https://github.com/google-health/medgemma/blob/main/notebooks)을 탐색하여 이 모델로 또 무엇을 할 수 있는지 알아보세요.


## 리소스 정리 (Clean up resources)

In [ ]:
# @markdown リ소스를 재활용(recycle)하고 발생할 수 있는 불필요한
# @markdown 지속적인 요금 청구를 피하기 위해 실험 모델들과 엔드포인트들을 삭제하세요.

# 모델 배포 해제(undeploy) 및 엔드포인트 삭제.
for endpoint in endpoints.values():
    endpoint.delete(force=True)

# 모델들 삭제.
for model in models.values():
    model.delete()

delete_bucket = False  # @param {type:"boolean"}
if delete_bucket:
    ! gsutil -m rm -r $BUCKET_NAME